# Detecção e Classificação de Lesões Mamárias em Ultrassonografia com YOLO11

Treinamento de um detector **YOLO11s** (Ultralytics) para **localizar e classificar** lesões **benignas** e **malignas** em imagens de ultrassom de mama, usando o dataset público [BUSI — *Breast Ultrasound Images Dataset*](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset) (Al-Dhabyani et al., 2020).

Notebook desenvolvido para rodar no **Kaggle** (GPU T4).

**Autores:** Juan José Gouvêa Cardenas e Felipe Ramirez Pereira Botero.

**Melhorias metodológicas em relação a versões anteriores:**
1. **Deduplicação** das imagens antes da divisão (o BUSI tem ~19% de duplicatas que causam *data leakage*).
2. **Divisão estratificada** em treino/validação/**teste** (70/15/15), com semente fixa.
3. Imagens **normais** tratadas como **exemplos negativos** (fundo), não como classe de detecção.
4. **Avaliação adequada a detectores** via `model.val()`: mAP@50, mAP@50–95 e precisão/revocação/F1 **por classe** — em vez de `classification_report` sobre a primeira caixa.

> ⚠️ **Aviso:** ferramenta de pesquisa acadêmica / prova de conceito. **Não** é dispositivo médico, não constitui diagnóstico e não substitui a avaliação de um profissional de saúde.

## 1. Instalação dos Pacotes

- **`ultralytics`**: biblioteca dos modelos YOLO (treino, validação e inferência), incluindo o YOLO11.
- **`opencv-python-headless`**: leitura/processamento de imagens e máscaras.
- **`pyyaml`**: leitura/escrita do arquivo de configuração do dataset.
- **`tqdm`**: barras de progresso.
- **`scikit-learn`**: divisão estratificada treino/validação/teste.

In [ ]:
# Instalação de pacotes necessários
!pip install ultralytics opencv-python-headless pyyaml tqdm scikit-learn --quiet

## 2. Importação das Bibliotecas

In [ ]:
import os
import glob
import shutil
import hashlib
from collections import Counter

import yaml
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import matplotlib.pyplot as plt

## 3. Diretório Base e Reprodutibilidade

Definimos o diretório de trabalho e uma **semente fixa** (`SEED`), usada em toda divisão aleatória para garantir reprodutibilidade.

In [ ]:
base_dir = '/kaggle/working/Breast_Ultrasound_Project'
os.makedirs(base_dir, exist_ok=True)

SEED = 42

## 4. Verificação da Disponibilidade do Dataset

Confirma que o BUSI está anexado ao notebook. No Kaggle, adicione o dataset *Breast Ultrasound Images Dataset* em **Add Input**.

In [ ]:
dataset_path = '/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT'
assert os.path.exists(dataset_path), (
    'Dataset não encontrado. Anexe o "Breast Ultrasound Images Dataset" ao notebook (Add Input).'
)
print(f'Dataset disponível em: {dataset_path}')

## 5. Catalogação das Imagens e Máscaras

Percorremos as três pastas (`benign`, `malignant`, `normal`) e montamos uma lista de registros com o caminho da imagem, da máscara (quando existir) e a classe.

In [ ]:
classes = ['benign', 'malignant', 'normal']
images_list = []

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    for image_file in glob.glob(os.path.join(cls_path, '*.png')):
        if '_mask' in image_file:
            continue
        mask_file = image_file.replace('.png', '_mask.png')
        images_list.append({
            'image_path': image_file,
            'mask_path': mask_file if os.path.exists(mask_file) else None,
            'class': cls,
        })

print('Total catalogado:', len(images_list))
print('Por classe:', Counter(it['class'] for it in images_list))

## 6. Deduplicação (mitigação de *data leakage*)

O BUSI contém imagens duplicadas documentadas (cerca de 19%; Pawlowska et al., 2023). Se duplicatas caírem em *splits* diferentes, há **vazamento de dados** e as métricas ficam infladas. Removemos duplicatas exatas comparando o **hash MD5 do conteúdo** de cada imagem, mantendo a primeira ocorrência.

In [ ]:
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

seen_hashes = set()
deduped = []
for it in tqdm(images_list, desc='Deduplicando'):
    h = file_hash(it['image_path'])
    if h in seen_hashes:
        continue
    seen_hashes.add(h)
    deduped.append(it)

removidas = len(images_list) - len(deduped)
print(f'Duplicatas removidas: {removidas}')
print('Após deduplicação:', len(deduped))
print('Por classe:', Counter(it['class'] for it in deduped))
images_list = deduped

## 7. Divisão Estratificada em Treino / Validação / Teste (70 / 15 / 15)

Dividimos **antes** de gerar rótulos e **antes** de qualquer aumento de dados, estratificando por classe para preservar a proporção benigno/maligno/normal em cada subconjunto. O conjunto de **teste** fica reservado (cego) para a avaliação final.

In [ ]:
labels = [it['class'] for it in images_list]

# 70% treino, 30% restante; depois 15%/15% (val/teste)
train_items, temp_items = train_test_split(
    images_list, test_size=0.30, random_state=SEED, stratify=labels
)
temp_labels = [it['class'] for it in temp_items]
val_items, test_items = train_test_split(
    temp_items, test_size=0.50, random_state=SEED, stratify=temp_labels
)

for nome, subset in [('treino', train_items), ('val', val_items), ('teste', test_items)]:
    print(f'{nome:7s}: {len(subset):3d}  ->  {dict(Counter(it["class"] for it in subset))}')

## 8. Geração dos Rótulos YOLO e Organização dos Diretórios

Para cada imagem com lesão, convertemos a **máscara de segmentação** na caixa delimitadora (`bounding box`) normalizada exigida pelo YOLO. Classes de detecção: **0 = benigno**, **1 = maligno**. Imagens **normais** recebem um arquivo de rótulo **vazio** (exemplo negativo / fundo). Cada subconjunto é gravado diretamente em `images/<split>` e `labels/<split>`.

In [ ]:
class_mapping = {'benign': 0, 'malignant': 1}

def mask_to_yolo_label(mask_path, width, height, class_id):
    """Converte a máscara binária na linha de rótulo YOLO. Retorna '' se vazia."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return ''
    coords = cv2.findNonZero(mask)
    if coords is None:
        return ''
    x, y, w, h = cv2.boundingRect(coords)
    x_c = (x + w / 2) / width
    y_c = (y + h / 2) / height
    return f'{class_id} {x_c:.6f} {y_c:.6f} {w / width:.6f} {h / height:.6f}\n'

def materializar(subset, split):
    img_dir = os.path.join(base_dir, 'images', split)
    lbl_dir = os.path.join(base_dir, 'labels', split)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    n_lesoes = 0
    for it in subset:
        img = cv2.imread(it['image_path'])
        if img is None:
            continue
        height, width = img.shape[:2]
        label = ''
        if it['class'] != 'normal' and it['mask_path']:
            label = mask_to_yolo_label(it['mask_path'], width, height, class_mapping[it['class']])
            if label:
                n_lesoes += 1
        stem = f"{it['class']}_{os.path.splitext(os.path.basename(it['image_path']))[0]}"
        shutil.copy(it['image_path'], os.path.join(img_dir, stem + '.png'))
        with open(os.path.join(lbl_dir, stem + '.txt'), 'w') as f:
            f.write(label)
    return n_lesoes

for nome, subset in [('train', train_items), ('val', val_items), ('test', test_items)]:
    n = materializar(subset, nome)
    print(f'{nome}: {len(subset)} imagens, {n} com lesão rotulada')

## 9. Verificação da Estrutura Final

In [ ]:
for split in ['train', 'val', 'test']:
    imgs = glob.glob(os.path.join(base_dir, 'images', split, '*.png'))
    lbls = glob.glob(os.path.join(base_dir, 'labels', split, '*.txt'))
    vazios = sum(1 for l in lbls if os.path.getsize(l) == 0)
    print(f'{split:5s}: {len(imgs)} imagens | {len(lbls)} rótulos | {vazios} negativos (normais)')

## 10. Carregamento do YOLO11s

Usamos a variante **`s` (small)** com pesos **pré-treinados em COCO** (transferência de aprendizado) — escolha adequada a um dataset de poucas centenas de imagens, evitando o *overfitting* de variantes maiores. Os pesos são baixados automaticamente pelo Ultralytics.

In [ ]:
model = YOLO('yolo11s.pt')

## 11. Arquivo YAML de Configuração

Define os caminhos de treino/validação/**teste** e as duas classes de detecção.

In [ ]:
yaml_content = {
    'path': base_dir,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': ['benigno', 'maligno'],
}
yaml_file_path = os.path.join(base_dir, 'breast_ultrasound.yaml')
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)
print('YAML criado:', yaml_file_path)

## 12. Treinamento

Aumento de dados **conservador** para ultrassom: rotações leves e espelhamento horizontal ajudam, enquanto distorções geométricas agressivas (*shear*/*perspective*) e espelhamento vertical são desabilitados por distorcerem a anatomia. *Early stopping* com `patience=25` evita corte prematuro. O treino só roda se ainda não houver um modelo salvo.

In [ ]:
trained_model_path = os.path.join(base_dir, 'best_model.pt')

if not os.path.exists(trained_model_path):
    print('Iniciando o treinamento...')
    model.train(
        data=yaml_file_path,
        epochs=100,
        imgsz=640,
        batch=16,
        workers=4,
        patience=25,
        seed=SEED,
        optimizer='auto',
        degrees=10.0,    # rotação leve (benéfica em US)
        fliplr=0.5,      # espelhamento horizontal
        flipud=0.0,      # sem espelhamento vertical
        translate=0.1,
        scale=0.3,
        shear=0.0,       # sem cisalhamento
        perspective=0.0, # sem perspectiva
    )
    shutil.copy('runs/detect/train/weights/best.pt', trained_model_path)
    print('Modelo salvo em', trained_model_path)
else:
    print('Modelo treinado já existe. Pulando o treinamento.')

## 13. Carregamento do Modelo Treinado

In [ ]:
model = YOLO(trained_model_path)

## 14. Avaliação de Detecção (mAP e métricas por classe)

Avaliação correta com `model.val()`, que casa predições e verdade por **IoU**. Reportamos **mAP@50**, **mAP@50–95** e **precisão/revocação/F1 por classe** no conjunto de **teste**. A revocação equivale à **sensibilidade** — métrica prioritária na classe *maligno*. O Ultralytics também salva a matriz de confusão e a curva *precision–recall* na pasta de resultados.

In [ ]:
metrics = model.val(split='test')

print('==== Métricas globais (teste) ====')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50-95 : {metrics.box.map:.4f}')
print(f'Precisão  : {metrics.box.mp:.4f}')
print(f'Revocação : {metrics.box.mr:.4f}')

print('\n==== Métricas por classe ====')
print(f'{"classe":10s} {"P":>7s} {"R":>7s} {"F1":>7s} {"AP@50":>7s} {"AP@50-95":>9s}')
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f'{metrics.names[c]:10s} {p:7.3f} {r:7.3f} {f1:7.3f} {ap50:7.3f} {ap:9.3f}')

print('\nResultados e gráficos salvos em:', metrics.save_dir)

## 15. Taxa de Falsos Positivos nas Imagens Normais

Como as imagens normais são exemplos negativos (sem lesão), o esperado é **nenhuma detecção**. Medimos quantas geram detecção espúria — indicador direto da propensão a falsos positivos.

In [ ]:
normais_teste = glob.glob(os.path.join(base_dir, 'images', 'test', 'normal_*.png'))
fp = 0
for img_path in normais_teste:
    r = model(img_path, verbose=False)[0]
    if r.boxes is not None and len(r.boxes) > 0:
        fp += 1

total = len(normais_teste)
taxa = (fp / total * 100) if total else 0.0
print(f'Imagens normais (teste): {total}')
print(f'Com detecção espúria   : {fp} ({taxa:.1f}%)')

## 16. Inferência Qualitativa

Visualização de algumas detecções no conjunto de teste, com as caixas e o grau de confiança.

In [ ]:
from IPython.display import Image, display

def mostrar_exemplos(model, split_dir, prefixo, n=3):
    imgs = sorted(glob.glob(os.path.join(split_dir, f'{prefixo}_*.png')))[:n]
    out_dir = os.path.join(base_dir, 'inference_results')
    os.makedirs(out_dir, exist_ok=True)
    for img_path in imgs:
        r = model(img_path, verbose=False)[0]
        save_path = os.path.join(out_dir, os.path.basename(img_path))
        cv2.imwrite(save_path, r.plot())
        display(Image(filename=save_path))

test_dir = os.path.join(base_dir, 'images', 'test')
print('Exemplos malignos:');  mostrar_exemplos(model, test_dir, 'malignant')
print('Exemplos benignos:');  mostrar_exemplos(model, test_dir, 'benign')
print('Exemplos normais:');   mostrar_exemplos(model, test_dir, 'normal')